# N080 · DFS、连通分量与环检测

**目标：** 用访问状态区分未访问、搜索中和已完成。

**先修：** N062, N067, N079。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 递归/迭代DFS；洪泛；分量；有向三色法；无向父边。

## 从问题到算法

无向连通分量可从每个未访问顶点启动DFS。判断有向环不能仅看“是否访问过”，必须区分正在当前路径上的灰点与已完成的黑点；只有指向灰点的边才直接证明回到祖先。克隆图先创建节点映射，再复制边，才能处理环和共享邻居。

## 最小实现

**本章接口：** `count_components(n, edges)`、`num_islands(grid)`、`has_directed_cycle(adj)`、`clone_graph(node)`

In [1]:
from collections import deque

def count_components(n,edges):
    adj=[[] for _ in range(n)]
    for u,v in edges: adj[u].append(v); adj[v].append(u)
    seen=set(); count=0
    for start in range(n):
        if start in seen: continue
        count+=1; seen.add(start); stack=[start]
        while stack:
            for v in adj[stack.pop()]:
                if v not in seen: seen.add(v); stack.append(v)
    return count

def num_islands(grid):
    if not grid or not grid[0]: return 0
    rows,cols=len(grid),len(grid[0]); seen=set(); count=0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c]!='1' or (r,c) in seen: continue
            count+=1; stack=[(r,c)]; seen.add((r,c))
            while stack:
                x,y=stack.pop()
                for nx,ny in [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]:
                    if 0<=nx<rows and 0<=ny<cols and grid[nx][ny]=='1' and (nx,ny) not in seen:
                        seen.add((nx,ny)); stack.append((nx,ny))
    return count

def has_directed_cycle(adj):
    color=[0]*len(adj)
    for start in range(len(adj)):
        if color[start]: continue
        color[start]=1; stack=[(start,iter(adj[start]))]
        while stack:
            u,neighbors=stack[-1]
            try: v=next(neighbors)
            except StopIteration:
                color[u]=2; stack.pop(); continue
            if color[v]==1: return True
            if color[v]==0: color[v]=1; stack.append((v,iter(adj[v])))
    return False

class GraphNode:
    def __init__(self,val=0,neighbors=None): self.val=val; self.neighbors=[] if neighbors is None else neighbors

def clone_graph(node):
    if node is None: return None
    mapping={node:GraphNode(node.val)}; queue=deque([node])
    while queue:
        u=queue.popleft()
        for v in u.neighbors:
            if v not in mapping: mapping[v]=GraphNode(v.val); queue.append(v)
            mapping[u].neighbors.append(mapping[v])
    return mapping[node]

## 正确性、前提与复杂度

一次搜索到达且仅到达起点所在分量。灰点仍在DFS祖先链，回到灰点闭合有向环；到黑点可能只是两条路径汇合，不能据此判环。映射按对象身份区分值相同的不同节点。

**代价：** O(V+E)时间、O(V)访问状态；克隆输出O(V+E)，岛屿O(RC)。迭代DFS支持深图。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['邻接表','是否有向环'],[([[1],[2],[]],has_directed_cycle([[1],[2],[]])),([[1],[2],[0]],has_directed_cycle([[1],[2],[0]]))])
show_table(['图顶点数','边','连通分量'],[(5,[(0,1),(1,2),(3,4)],count_components(5,[(0,1),(1,2),(3,4)]))])

邻接表,是否有向环
"[[1], [2], []]",False
"[[1], [2], [0]]",True


图顶点数,边,连通分量
5,"[(0, 1), (1, 2), (3, 4)]",2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert count_components(0,[])==0 and count_components(4,[])==4
assert has_directed_cycle([[0]]) and not has_directed_cycle([[1,2],[2],[]])
assert num_islands([list('110'),list('001')])==2
x,y=GraphNode(7),GraphNode(7); x.neighbors=[x,y]; y.neighbors=[x]
a=clone_graph(x)
assert a is not x and a.val==7 and a.neighbors[0] is a
assert a.neighbors[1] is not y and a.neighbors[1].neighbors[0] is a
from itertools import combinations
n=4; pairs=list(combinations(range(n),2))
for mask in range(1<<len(pairs)):
    edges=[e for i,e in enumerate(pairs) if mask>>i&1]
    reach=[[i==j for j in range(n)] for i in range(n)]
    for u,v in edges: reach[u][v]=reach[v][u]=True
    for k in range(n):
        for i in range(n):
            for j in range(n): reach[i][j]|=reach[i][k] and reach[k][j]
    groups={tuple(j for j in range(n) if reach[i][j]) for i in range(n)}
    assert count_components(n,edges)==len(groups)
print('N080: 所有本章断言通过')

N080: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用迭代DFS避免深递归。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较有向与无向环检测条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 200. Number of Islands（canonical）
- 547. Number of Provinces（canonical）
- 133. Clone Graph（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。